# Diferencia observada, ATE, ATT, ATC, CATE

### Descripción de Conceptos de Inferencia Causal

A continuación se definen los conceptos de inferencia causal aplicados en este análisis:

1. **Diferencia Observada (Observed Difference)**:
   * **Qué es**: Es la diferencia simple entre el promedio del resultado ($Y$) del grupo que recibió el tratamiento ($T=1$) y el promedio del grupo de control ($T=0$).
   * **Limitación**: No es una estimación causal válida si hay sesgo de selección (es decir, si los grupos de tratamiento y control difieren de forma sistemática en otras variables que también afectan a la supervivencia).

2. **ATE (Average Treatment Effect - Efecto Promedio del Tratamiento)**:
   * **Qué es**: Es el impacto promedio que tendría el tratamiento si se aplicara a **toda la población** bajo estudio.
   * **Cálculo**: Es el promedio de la diferencia entre el resultado potencial con tratamiento y sin tratamiento para cada individuo: $\mathbb{E}[Y(1) - Y(0)]$.

3. **ATT (Average Treatment Effect on the Treated - Efecto Promedio en los Tratados)**:
   * **Qué es**: Es el impacto promedio del tratamiento únicamente en aquellos individuos que **efectivamente recibieron el tratamiento** ($T=1$).
   * **Cálculo**: $\mathbb{E}[Y(1) - Y(0) \mid T=1]$. Responde a la pregunta: *¿Cuánto se beneficiaron de viajar en Primera Clase aquellos que realmente lo hicieron?*

4. **ATC (Average Treatment Effect on the Control - Efecto Promedio en los Controles)**:
   * **Qué es**: Es el impacto promedio que habría tenido el tratamiento únicamente en aquellos que **no lo recibieron** ($T=0$).
   * **Cálculo**: $\mathbb{E}[Y(1) - Y(0) \mid T=0]$. Responde a la pregunta: *¿Cuánto habrían mejorado sus probabilidades de sobrevivir los pasajeros de clases inferiores si hubieran viajado en Primera Clase?*

5. **CATE (Conditional Average Treatment Effect - Efecto Promedio Condicional del Tratamiento)**:
   * **Qué es**: Es el efecto promedio del tratamiento condicionado a ciertas características o subgrupos de la población (como género o grupos de edad).
   * **Cálculo**: $\mathbb{E}[Y(1) - Y(0) \mid X=x]$. Permite identificar la heterogeneidad del efecto (por ejemplo, ver si el tratamiento es más efectivo en hombres que en mujeres, o en jóvenes que en adultos mayores).

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np

df = sns.load_dataset("titanic")

df = df[["survived", "pclass", "sex", "age", "sibsp", "parch"]].copy()

df["treatment"] = (df["pclass"] == 1).astype(int)

df = df.dropna().copy()

X = df[["sex", "age", "sibsp", "parch"]]
T = df["treatment"]
Y = df["survived"]

## Diferencia observada

In [ ]:
observed_difference = (
    df.loc[T == 1, "survived"].mean()
    - df.loc[T == 0, "survived"].mean()
)

print(f"Observed difference: {observed_difference:.3f}")

Observed difference: 0.338


## ATE

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

features = ["treatment", "sex", "age", "sibsp", "parch"]
categorical = ["sex"]
numeric = ["treatment", "age", "sibsp", "parch"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", StandardScaler(), numeric),
])

model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

model.fit(df[features], Y)

df_treated = df[features].copy()
df_control = df[features].copy()

df_treated["treatment"] = 1
df_control["treatment"] = 0

pred_treated = model.predict_proba(df_treated)[:, 1]
pred_control = model.predict_proba(df_control)[:, 1]

ate = np.mean(pred_treated - pred_control)

print(f"Estimated ATE: {ate:.3f}")

Estimated ATE: 0.349


## ATT y ATC

In [ ]:
att = np.mean(
    pred_treated[T.to_numpy() == 1]
    - pred_control[T.to_numpy() == 1]
)

atc = np.mean(
    pred_treated[T.to_numpy() == 0]
    - pred_control[T.to_numpy() == 0]
)

print(f"ATE: {ate:.3f}")
print(f"ATT: {att:.3f}")
print(f"ATC: {atc:.3f}")

ATE: 0.349
ATT: 0.325
ATC: 0.358


## CATE

In [ ]:
results = df[["sex", "age"]].copy()
results["effect"] = pred_treated - pred_control

results["age_group"] = pd.cut(
    results["age"],
    bins=[0, 18, 40, 60, np.inf],
    labels=["0-18", "19-40", "41-60", "60+"],
    include_lowest=True
)

cate_by_sex = results.groupby("sex", observed=True)["effect"].mean()
cate_by_age = results.groupby("age_group", observed=True)["effect"].mean()

print(cate_by_sex)
print(cate_by_age)

sex
female    0.281123
male      0.388675
Name: effect, dtype: float64
age_group
0-18     0.319182
19-40    0.369463
41-60    0.334480
60+      0.238231
Name: effect, dtype: float64


## Conclusión de los Análisis de Inferencia Causal

A partir de los cálculos realizados sobre el conjunto de datos del Titanic, donde el **tratamiento ($T$)** es haber viajado en **Primera Clase** y el **resultado ($Y$)** es la **supervivencia**, podemos extraer las siguientes conclusiones:

### 1. Comparación General (Diferencia Observada vs. ATE)
* **Diferencia Observada (~33.8%)**: La tasa de supervivencia de los pasajeros de Primera Clase es un **33.8% mayor** que la de las clases inferiores si solo comparamos los promedios directos de los datos.
* **ATE (Efecto Promedio del Tratamiento) (~34.9%)**: Al controlar por otras variables (género, edad, hermanos/esposos y padres/hijos) usando regresión logística, estimamos que si **absolutamente todos** los pasajeros hubieran viajado en Primera Clase, la probabilidad global de supervivencia habría aumentado en un **34.9%**.
* *Interpretación*: Dado que el ATE (34.9%) es muy cercano a la diferencia observada (33.8%), el sesgo de selección neto en la muestra general no altera radicalmente la magnitud de la estimación causal general, aunque el control por covariables nos da una estimación más precisa y libre de confusores.

### 2. Impacto Diferencial (ATT vs. ATC)
* **ATT (Efecto en los Tratados) (~32.5%)**: Los pasajeros que **realmente viajaron en Primera Clase** incrementaron su probabilidad de sobrevivir en un **32.5%** gracias a estar en esa clase.
* **ATC (Efecto en los Controles) (~35.8%)**: Si los pasajeros que viajaron en **clases inferiores (segunda y tercera)** hubieran sido asignados a Primera Clase, su probabilidad de sobrevivir habría aumentado en un **35.8%**.
* *Interpretación*: El impacto potencial de la Primera Clase era aún mayor para quienes no viajaron en ella (ATC > ATT). Esto puede deberse a que el grupo de control contenía perfiles de pasajeros (por ejemplo, más hombres jóvenes o familias numerosas) que en condiciones normales tuvieron tasas de supervivencia muy bajas y se habrían beneficiado enormemente de la prioridad de evacuación de Primera Clase.

### 3. Heterogeneidad del Efecto (CATE)
El efecto de viajar en Primera Clase no fue el mismo para todos los subgrupos:

* **Por Género**:
  * **Hombres (~38.9%)** vs. **Mujeres (~28.1%)**:
  * *Interpretación*: El "efecto tratamiento" de la Primera Clase fue más drástico para los hombres. Esto ocurre porque las mujeres en general tenían una alta prioridad de evacuación ("mujeres y niños primero") independientemente de la clase, mientras que para los hombres, el estatus social y la ubicación en Primera Clase marcaron una diferencia de vida o muerte mucho más pronunciada.

* **Por Edad**:
  * **19-40 años (~36.9%)** y **0-18 años (~31.9%)** presentan los mayores beneficios de viajar en Primera Clase.
  * **Mayores de 60 años (~23.8%)** experimentan el menor beneficio relativo.
  * *Interpretación*: Los adultos jóvenes y de mediana edad en clases inferiores tenían muy pocas probabilidades de salvarse al no ser prioritarios, por lo que estar en Primera Clase les daba una ventaja enorme. En cambio, para los adultos mayores, las limitaciones físicas u otros factores reducían el beneficio neto de supervivencia que otorgaba la Primera Clase en comparación con los grupos más jóvenes.